In [1]:
import sympy as sp
from sympy import sqrt, eye, Matrix, binomial, S
from sympy.physics.quantum.trace import Tr
from sympy.physics.quantum.tensorproduct import TensorProduct
import dill

def amplitude_damping_kraus(dim, symbol):
    """
    Generate the Kraus operators for the amplitude damping channel
    """
    kraus_ops = []
    for k in range(dim):
        K = Matrix.zeros(dim, dim)
        for n in range(k, dim):
            amp = (
                sqrt(binomial(n, k))
                * ((1 - symbol) ** ((n - k) / S(2)))
                * (symbol ** (k / S(2)))
            )
            K[n - k, n] = amp
        kraus_ops.append(K)

    return kraus_ops

# Define ion-excitation strength
q = sp.Symbol("q")

# Define ion state vector
ket_0_ion = sp.Matrix([1, 0])
ket_1_ion = sp.Matrix([0, 1])

bra_0_ion = ket_0_ion.T
bra_1_ion = ket_1_ion.T

# Define photon state vector
ket_0_photon = sp.Matrix([1, 0, 0])
ket_1_photon = sp.Matrix([0, 1, 0])
ket_2_photon = sp.Matrix([0, 0, 1])

bra_0_photon = ket_0_photon.T
bra_1_photon = ket_1_photon.T
bra_2_photon = ket_2_photon.T

# Define identity operator
id_ion = eye(2)
id_photon = eye(3)
id_ion_all = eye(8)

# Define remote loss
p_r = sp.Symbol("p_r")

# Define local loss
p_l = sp.Symbol("p_l")

# Define Kraus operators for the remote loss
K_ops_remote = amplitude_damping_kraus(3, p_r)

# Define Kraus operators for the ion emission
K_ops_ion_emission = amplitude_damping_kraus(3, p_l)
ket_W = (1 / sqrt(3)) * (
    TensorProduct(ket_0_ion, ket_0_ion, ket_1_ion)
    + TensorProduct(ket_0_ion, ket_1_ion, ket_0_ion)
    + TensorProduct(ket_1_ion, ket_0_ion, ket_0_ion)
)

def normalize(dm):
    return dm / Tr(dm)

def remote_click(dm):
    # The input mode corresponds to detector mode |100>
    t_100 = (1 / sqrt(3)) * (
        1
        * TensorProduct(
            id_ion, ket_1_photon, id_ion, ket_0_photon, id_ion, ket_0_photon
        )
        + 1
        * TensorProduct(
            id_ion, ket_0_photon, id_ion, ket_1_photon, id_ion, ket_0_photon
        )
        + 1
        * TensorProduct(
            id_ion, ket_0_photon, id_ion, ket_0_photon, id_ion, ket_1_photon
        )
    )

    dm_100 = t_100.T @ dm @ t_100

    return dm_100
sp_ion_photon_ket = sqrt(1 - q) * TensorProduct(ket_0_ion, ket_0_photon) + sqrt(
    q
) * TensorProduct(ket_1_ion, ket_1_photon)
sp_ion_photon_dm = sp_ion_photon_ket @ sp_ion_photon_ket.T
sp_ion_photon_dm_damp = sp.zeros(sp_ion_photon_dm.shape[0])

# ----- ion emission damping ---
for i in range(len(K_ops_ion_emission)):
    sp_ion_photon_dm_damp += (
        TensorProduct(id_ion, K_ops_ion_emission[i])
        @ sp_ion_photon_dm
        @ TensorProduct(id_ion, K_ops_ion_emission[i].T)
    )

sp_joint_dm = TensorProduct(
    sp_ion_photon_dm_damp, sp_ion_photon_dm_damp, sp_ion_photon_dm_damp
)

sp_joint_dm_damp1 = sp.zeros(sp_joint_dm.shape[0])
sp_joint_dm_damp2 = sp.zeros(sp_joint_dm.shape[0])
sp_joint_dm_damp3 = sp.zeros(sp_joint_dm.shape[0])

# Damp the first photon
for i in range(len(K_ops_remote)):
    sp_joint_dm_damp1 += (
        TensorProduct(id_ion, K_ops_remote[i], id_ion, id_photon, id_ion, id_photon)
        @ sp_joint_dm
        @ TensorProduct(id_ion, K_ops_remote[i].T, id_ion, id_photon, id_ion, id_photon)
    )

# Damp the second photon
for i in range(len(K_ops_remote)):
    sp_joint_dm_damp2 += (
        TensorProduct(id_ion, id_photon, id_ion, K_ops_remote[i], id_ion, id_photon)
        @ sp_joint_dm_damp1
        @ TensorProduct(id_ion, id_photon, id_ion, K_ops_remote[i].T, id_ion, id_photon)
    )

# Damp the third photon
for i in range(len(K_ops_remote)):
    sp_joint_dm_damp3 += (
        TensorProduct(id_ion, id_photon, id_ion, id_photon, id_ion, K_ops_remote[i])
        @ sp_joint_dm_damp2
        @ TensorProduct(id_ion, id_photon, id_ion, id_photon, id_ion, K_ops_remote[i].T)
    )

ion_dm = remote_click(sp_joint_dm_damp3)
prob_click = sp.simplify(3 * Tr(ion_dm))
ion_dm = normalize(ion_dm)
get_prob_click = sp.lambdify([q, p_r, p_l], prob_click)
get_ion_dm = sp.lambdify([q, p_r, p_l], ion_dm)

with open("atom_based_prob_click.dill", "wb") as file:
    dill.dump(prob_click, file)

with open("atom_based_ion_dm.dill", "wb") as file:
    dill.dump(ion_dm, file)